# 02 - Evaluation Scoring (Multi-Endpoint Load Balanced)

**Configuration**:
- **2x Glider Endpoints**: `localhost:8807`, `localhost:8808`
- **2x Molmo Endpoints**: `localhost:8806`, `localhost:8809`
- **Pipeline**: Distributes requests round-robin across these endpoints.
- **Parallelism**: High concurrency to maximize throughput.

In [1]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths - all ares notebooks are in artemis_final/notebooks/ares/
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")

# Cell 1: Setup
%load_ext autoreload
%autoreload 2


import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s', datefmt='%H:%M:%S')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)

In [2]:
# Cell 2: Database Connection
from sqlalchemy import text
from ares.db.connection import get_engine

engine = get_engine()
print("Connected to DB")

# Apply migrations
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_score FLOAT"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_rank_group INTEGER"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_raw TEXT"))
    conn.commit()
print("Migrations applied")

Connected to DB
Migrations applied


In [3]:
# Cell 3: Configure Endpoints
from inference_engine.runners import OpenAIStyleRunner
from inference_engine.config import ModelEndpoint

# Your current vLLM setup:
# - Glider: ports 8805, 8807
# - Llama Scout: ports 8806, 8808

endpoints = [
    # Glider (text-only LLM judge)
    ModelEndpoint(
        name="glider-1",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8805/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="glider-2",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8807/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    # Llama Scout (VLM judge with image)
    ModelEndpoint(
        name="vlm-judge-1",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8806/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="vlm-judge-2",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8808/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
]

runner = OpenAIStyleRunner(
    models=endpoints,
    request_timeout_s=180,
    max_workers=64
)

print(f"Configured {len(endpoints)} endpoints:")
for ep in endpoints:
    print(f"  - {ep.name}: {ep.base_url}")

Configured 4 endpoints:
  - glider-1: http://localhost:8805/v1
  - glider-2: http://localhost:8807/v1
  - vlm-judge-1: http://localhost:8806/v1
  - vlm-judge-2: http://localhost:8808/v1


In [4]:
# Cell 4: Initialize Pipeline
from ares.evaluation.router_eval_pipeline import RouterEvalPipeline

pipeline = RouterEvalPipeline(
    engine=engine,
    runner=runner,
    glider_model_names=["glider-1", "glider-2"],
    vlm_judge_model_names=["vlm-judge-1", "vlm-judge-2"],
    tracker_path="eval_progress.json",
    use_glider=True,
    use_vlm_judge=True,
)

print("Pipeline initialized!")
print(f"Glider models: {pipeline.glider_model_names}")
print(f"VLM Judge models: {pipeline.vlm_judge_model_names}")

Pipeline initialized!
Glider models: ['glider-1', 'glider-2']
VLM Judge models: ['vlm-judge-1', 'vlm-judge-2']


In [ ]:
# Cell 5: Run Evaluation
# This will:
# 1. Load samples per source_config
# 2. Compute static metrics (exact match, F1, etc.)
# 3. Compute confidence scores
# 4. Run Glider (text evaluator) on 2 GPUs
# 5. Run Llama Scout (VLM judge with image) on 2 GPUs
# 6. Write all results to vlm_evaluations table

# Reset ALL progress
# pipeline.reset_progress()

pipeline.evaluate_all(
    batch_size=50,
    force=False,             # Set True to recompute all
    max_parallel_configs=4,  # Process 2 source_configs at a time
    split=None,              # Filter: 'train', 'val', 'test', or None
)

07:13:04 | EVAL_PIPELINE   | Progress reset: ALL
07:13:04 | EVAL_PIPELINE   | ============================================================
07:13:04 | EVAL_PIPELINE   | EVALUATION PIPELINE
07:13:04 | EVAL_PIPELINE   | ============================================================
07:13:04 | EVAL_PIPELINE   | Source configs: 48
07:13:04 | EVAL_PIPELINE   | Previous progress: 0 samples
07:13:04 | EVAL_PIPELINE   | Use Glider: True, Use VLM Judge: True
07:13:04 | EVAL_PIPELINE   | Force recompute: True
07:13:04 | EVAL_PIPELINE   | Split filter: ALL
07:13:04 | EVAL_PIPELINE   | ============================================================


[ai2d]:   0%|          | 0/1260 [00:00<?, ?s/s]

[chartqa]:   0%|          | 0/1270 [00:00<?, ?s/s]

[chart2text]:   0%|          | 0/1310 [00:00<?, ?s/s]

[aokvqa]:   0%|          | 0/1310 [00:00<?, ?s/s]

...

11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_40_c5a9edfa: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | Computed static metrics for 247 responses
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_297_c583ad0d: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_301_7c09d9cd: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_287_543a5137: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_29_dda2cd16: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | Computed confidence for 247 responses
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_309_389daac9: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_37_af3074f2: Failed after 3 retries: Connection error.
11:35:28 | EVAL_PIPELINE   | VLM Judge error for vqarad_30_e36a2cc6: Failed after

[vsr]:   0%|          | 0/1068 [00:00<?, ?s/s]

11:36:15 | EVAL_PIPELINE   | Computed static metrics for 248 responses
11:36:15 | EVAL_PIPELINE   | Computed static metrics for 246 responses
11:36:15 | EVAL_PIPELINE   | Computed confidence for 248 responses
11:36:15 | EVAL_PIPELINE   | Computed confidence for 246 responses
11:36:20 | EVAL_PIPELINE   | VLM Judge error for vqav2_469_fcf929e4: Failed after 3 retries: Connection error.
11:36:20 | EVAL_PIPELINE   | VLM Judge error for vsr_259_70e9cd7d: Failed after 3 retries: Connection error.
11:36:20 | EVAL_PIPELINE   | VLM Judge error for vqav2_215_2f806ed5: Failed after 3 retries: Connection error.
11:36:21 | EVAL_PIPELINE   | VLM Judge error for vqav2_360_466e3131: Failed after 3 retries: Connection error.
11:36:21 | EVAL_PIPELINE   | VLM Judge error for vqav2_148_419f2697: Failed after 3 retries: Connection error.
11:36:21 | EVAL_PIPELINE   | VLM Judge error for vqav2_699_918b2545: Failed after 3 retries: Connection error.
11:36:21 | EVAL_PIPELINE   | VLM Judge error for vqav2_544_2

[websight]:   0%|          | 0/1069 [00:00<?, ?s/s]

11:38:11 | EVAL_PIPELINE   | Computed static metrics for 248 responses
11:38:11 | EVAL_PIPELINE   | Computed confidence for 248 responses
11:38:12 | EVAL_PIPELINE   | Computed static metrics for 248 responses
11:38:12 | EVAL_PIPELINE   | Computed confidence for 248 responses
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_320_2d97da7b: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_34_deb741ee: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_188_9dae42ad: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_643_4a6d6a96: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_309_c694f6d7: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_27_58dff25e: Failed after 3 retries: Connection error.
11:38:16 | EVAL_PIPELINE   | VLM Judge error for vsr_315_7dc02a3a: Faile

In [6]:
# Cell 6: Verify Results
import pandas as pd

query = """
SELECT 
    r.model_name,
    COUNT(*) as total,
    ROUND(AVG(r.score_exact_match_normalized)::numeric, 3) as avg_em,
    ROUND(AVG(r.score_f1)::numeric, 3) as avg_f1,
    ROUND(AVG(e.glider_score)::numeric, 2) as avg_glider,
    ROUND(AVG(e.judge_molmo_score)::numeric, 2) as avg_vlm_judge,
    ROUND(AVG(e.judge_molmo_rank_group)::numeric, 2) as avg_rank
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
GROUP BY r.model_name
ORDER BY avg_vlm_judge DESC NULLS LAST
"""

results_df = pd.read_sql(query, engine)
print("Results by Model:")
results_df

Results by Model:


,model_name,total,avg_em,avg_f1,avg_glider,avg_vlm_judge,avg_rank
0,qwen3_vl_8b_thinking,55723,0.000,0.108,1.27,8.28,1.51
1,qwen2_5_vl_7b,55728,0.224,0.428,1.32,7.79,1.54
2,gemma_3_27b,55721,0.141,0.322,1.18,7.39,1.60
3,qwen2_5_vl_3b,55729,0.253,0.447,0.99,5.68,1.99
4,deepseek_ocr,46563,0.003,0.080,0.39,2.20,2.87


In [ ]:
# Cell 7: Check Coverage
coverage_query = """
SELECT 
    COUNT(*) as total_responses,
    SUM(CASE WHEN r.score_exact_match IS NOT NULL THEN 1 ELSE 0 END) as has_static,
    SUM(CASE WHEN r.confidence_score IS NOT NULL THEN 1 ELSE 0 END) as has_confidence,
    SUM(CASE WHEN e.glider_score IS NOT NULL THEN 1 ELSE 0 END) as has_glider,
    SUM(CASE WHEN e.judge_molmo_score IS NOT NULL THEN 1 ELSE 0 END) as has_vlm_judge
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
"""

coverage_df = pd.read_sql(coverage_query, engine)
print("Metric Coverage:")
coverage_df.T